# 問題
問題76のモデル学習をGPU上で実行せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [1]:
# 単語埋め込み語彙の作成
import numpy as np
from gensim.models import KeyedVectors
import torch

model = KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary=True)
vocab = list(model.key_to_index.keys())
d_emb = model.vector_size
V = len(vocab) + 1

# 埋め込み行列の初期化
E = np.zeros((V, d_emb), dtype=np.float32)

# インデックス対応表
word2id = {'<PAD>': 0}
id2word = {0: '<PAD>'}

# 行列にベクトルを格納
for i, word in enumerate(vocab, start=1):
    E[i] = model[word]
    word2id[word] = i
    id2word[i] = word

In [2]:
import torch

def sst_build_answer_batch(path: str):
    """
    SST-2のTSVを読み込み、
      - 文章→単語分割
      - word2idでID列に変換（辞書にない語は除外）
      - 最長系列長に合わせて0埋めパディング
      - トークン列の長い順にソート
      - 最終的に input_ids と label をそれぞれTensorとしてまとめて返す
    """
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            row = raw.strip().split("\t")
            if len(row) < 2:
                continue
            label = row[1]
            if label not in ("0", "1"):
                continue
            text = row[0]
            tokens = text.strip().split()
            examples.append({"text": text, "label": int(label), "tokens": tokens})

    # ID化
    kept_examples = []
    for ex in examples:
        ids = [word2id[w] for w in ex["tokens"] if w in word2id]
        if ids:
            ex["input_ids"] = ids
            kept_examples.append(ex)

    # --- 長い順にソート ---
    kept_examples = sorted(kept_examples, key=lambda x: len(x["input_ids"]), reverse=True)

    # --- パディング ---
    max_len = len(kept_examples[0]["input_ids"])
    padded_ids = []
    labels = []

    for ex in kept_examples:
        ids = ex["input_ids"]
        padded = ids + [0] * (max_len - len(ids))
        padded_ids.append(padded)
        labels.append([float(ex["label"])])  # [[1.], [0.], ...]

    # --- Tensor化 ---
    input_tensor = torch.tensor(padded_ids, dtype=torch.long)
    label_tensor = torch.tensor(labels, dtype=torch.float)

    # --- dict形式で返す ---
    batch = {
        "input_ids": input_tensor,
        "label": label_tensor
    }

    return batch
# --- pathの準備 ---
path_dev = "./SST-2/dev.tsv"
path_train = "./SST-2/train.tsv"
dev_71 = sst_build_answer_batch(path_dev)
train_71 = sst_build_answer_batch(path_train)

In [3]:
import torch
import torch.nn.functional as F
import numpy as np

def build_avg_features(batch, E, pad_id=0):
    """
    batch: {"input_ids": LongTensor (N, L), "label": FloatTensor (N, 1)}
    E    : (|V|, d)  埋め込み行列（torch.Tensor or np.ndarray）
    戻り値: X (N, d), y (N,)
    """
    # E を torch.Tensor(float32) に揃える
    if isinstance(E, np.ndarray):
        E = torch.tensor(E, dtype=torch.float32)
    elif not torch.is_tensor(E):
        raise TypeError("E must be a torch.Tensor or np.ndarray")
    if E.dtype != torch.float32:
        E = E.float()

    ids = batch["input_ids"]            # (N, L) long
    y   = batch["label"].squeeze(1)     # (N,) float

    # (N, L, d) に埋め込み
    emb = F.embedding(ids, E)           # = E[ids] と同等

    # PADを無視して平均
    mask = (ids != pad_id)              # (N, L) bool
    lens = mask.sum(dim=1, keepdim=True).clamp_min(1)  # (N, 1)

    emb_sum = (emb * mask.unsqueeze(-1).float()).sum(dim=1)  # (N, d)
    X = emb_sum / lens.float()                                # (N, d)

    return X, y

X_train, y_train = build_avg_features(train_71, E)
X_dev, y_dev = build_avg_features(dev_71,   E)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# 1) デバイス選択（NVIDIAならcuda、Apple Siliconならmps）
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)

# --- ここは既存の前処理で作ったテンソルを想定 ---
# X_train: (N_train, d), y_train: (N_train,) 0/1
# X_dev  : (N_dev, d),   y_dev  : (N_dev,)   0/1
# 例: すでにCPU上にあるとする
# X_train, y_train, X_dev, y_dev = ...

# 2) データローダ（ミニバッチ学習）
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    train_ds, batch_size=64, shuffle=True,
    pin_memory=torch.cuda.is_available()  # CUDA時だけpin_memory有効化
)

# 3) モデル作成＆GPUへ
model = nn.Linear(X_train.size(1), 1).to(device)  # (d -> 1)
crit  = nn.BCEWithLogitsLoss()
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 4) 学習ループ（GPU上で順伝播・逆伝播）
model.train()
for epoch in range(10):
    for xb, yb in train_loader:
        # バッチをGPUへ
        xb = xb.to(device)                 # (B, d)
        yb = yb.float().to(device)         # (B,)
        opt.zero_grad()
        logit = model(xb).squeeze(1)       # (B,)
        loss  = crit(logit, yb)
        loss.backward()
        opt.step()

# 5) 開発セットで正解率
model.eval()
with torch.no_grad():
    xd = X_dev.to(device)                  # (N_dev, d)
    yd = y_dev.long().to(device)           # (N_dev,)
    prob = torch.sigmoid(model(xd).squeeze(1))   # (N_dev,)
    pred = (prob >= 0.5).long()                  # 0/1
    acc  = (pred == yd).float().mean().item()

print(f"dev acc: {acc:.3f}")

device: mps
dev acc: 0.798
